In [ ]:
# Resume RAG — Retrieval Experiments & Analysis

This notebook evaluates the resume/job matching pipeline using the committed
30-resume dataset.

Experiments:
1. Inspect ingested resume chunks and metadata
2. Run sample job descriptions
3. Measure retrieval accuracy
4. Measure embedding, vector-query, and end-to-end latency
5. Analyze strengths, limitations, and an improvement

In [26]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

# If the notebook is being launched from the project root
# this should point to the resume-rag-job-matcher directory.
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("C:/Users/aswin/Documents/Airtribe/projects/resume-rag-job-matcher/resume-rag-job-matcher")

sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd())
print("src exists:", (PROJECT_ROOT / "src").exists())
print("src import path added:", str(PROJECT_ROOT) in sys.path)

from src.retrieval.vector_store import VectorStore

print("src import: OK")

Project root: C:\Users\aswin\Documents\Airtribe\projects\resume-rag-job-matcher\resume-rag-job-matcher
Working directory: C:\Users\aswin\Documents\Airtribe\projects\resume-rag-job-matcher\resume-rag-job-matcher
src exists: True
src import path added: True
src import: OK


In [27]:
from src.retrieval.vector_store import VectorStore

store = VectorStore()

print("Vector store collection:", store.collection.name)
print("Total indexed chunks:", store.collection.count())

sample = store.collection.get(
    limit=5,
    include=["documents", "metadatas"],
)

print("\nSample chunks:\n")

for i, (document, metadata) in enumerate(
    zip(sample["documents"], sample["metadatas"]),
    start=1,
):
    print("=" * 70)
    print(f"Sample {i}")
    print("-" * 70)
    print("Candidate:", metadata.get("candidate_name"))
    print("Resume:", metadata.get("resume_path"))
    print("Section:", metadata.get("section"))
    print("Experience:", metadata.get("experience_years"))
    print("Skills:", metadata.get("skills"))
    print("\nChunk:")
    print(document[:500])
    print()

Vector store collection: resume_chunks
Total indexed chunks: 157

Sample chunks:

Sample 1
----------------------------------------------------------------------
Candidate: Rohan Sharma
Resume: data/resumes/resume_01.txt
Section: PROFESSIONAL SUMMARY
Experience: 8.0
Skills: Java, Go, Python, SQL, Spring Boot, gRPC, Hibernate, FastAPI, PostgreSQL, MongoDB, Redis, Cassandra, AWS (EC2, S3, SQS), Docker, Kubernetes, Kafka, Git, CI/CD

Chunk:
Results-driven Senior Backend Engineer with 8+ years of core software development experience building high-throughput microservices and distributed systems. Proven track record at Google and enterprise financial tech startups in optimizing database performance, implementing event-driven architectures, and leading backend engineering teams.

Sample 2
----------------------------------------------------------------------
Candidate: Rohan Sharma
Resume: data/resumes/resume_01.txt
Section: TECHNICAL SKILLS
Experience: 8.0
Skills: Java, Go, Python, SQL, Spr

In [28]:
from src.retrieval.vector_store import VectorStore

store = VectorStore()

print("Vector store collection:", store.collection.name)
print("Total indexed chunks:", store.collection.count())

sample = store.collection.get(
    limit=5,
    include=["documents", "metadatas"],
)

print("\nSample chunks:\n")

for i, (document, metadata) in enumerate(
    zip(sample["documents"], sample["metadatas"]),
    start=1,
):
    print("=" * 70)
    print(f"Sample {i}")
    print("-" * 70)
    print("Candidate:", metadata.get("candidate_name"))
    print("Resume:", metadata.get("resume_path"))
    print("Section:", metadata.get("section"))
    print("Experience:", metadata.get("experience_years"))
    print("Skills:", metadata.get("skills"))
    print("\nChunk:")
    print(document[:500])
    print()

Vector store collection: resume_chunks
Total indexed chunks: 157

Sample chunks:

Sample 1
----------------------------------------------------------------------
Candidate: Rohan Sharma
Resume: data/resumes/resume_01.txt
Section: PROFESSIONAL SUMMARY
Experience: 8.0
Skills: Java, Go, Python, SQL, Spring Boot, gRPC, Hibernate, FastAPI, PostgreSQL, MongoDB, Redis, Cassandra, AWS (EC2, S3, SQS), Docker, Kubernetes, Kafka, Git, CI/CD

Chunk:
Results-driven Senior Backend Engineer with 8+ years of core software development experience building high-throughput microservices and distributed systems. Proven track record at Google and enterprise financial tech startups in optimizing database performance, implementing event-driven architectures, and leading backend engineering teams.

Sample 2
----------------------------------------------------------------------
Candidate: Rohan Sharma
Resume: data/resumes/resume_01.txt
Section: TECHNICAL SKILLS
Experience: 8.0
Skills: Java, Go, Python, SQL, Spr

In [29]:
## 2. Job Matching Experiments

Run several sample job descriptions through the complete job matching pipeline and compare the top candidates.

SyntaxError: invalid syntax (987752062.py, line 3)

In [30]:
from pathlib import Path

from src.job_matcher import load_job_description, match_job


job_description_dir = Path("data/job_descriptions")
job_files = sorted(job_description_dir.glob("*.txt"))[:5]

print("=" * 80)
print("JOB MATCHING EXPERIMENTS")
print("=" * 80)

for jd_file in job_files:
    print()
    print("=" * 80)
    print(f"JOB DESCRIPTION: {jd_file.name}")
    print("=" * 80)

    jd = load_job_description(str(jd_file))
    result = match_job(jd)

    print(f"Candidates matched: {result['total_candidates']}")
    print()

    for rank, candidate in enumerate(
        result["top_matches"][:3],
        start=1,
    ):
        print(
            f"{rank}. "
            f"{candidate['candidate_name']} | "
            f"Score: {candidate['match_score']} | "
            f"Experience: {candidate['experience_years']} years"
        )

        print(
            "   Skills:",
            ", ".join(candidate["matched_skills"])
        )

        print(
            "   Reasoning:",
            candidate["reasoning"]
        )

JOB MATCHING EXPERIMENTS

JOB DESCRIPTION: job_description_01.txt
Candidates matched: 1

1. Jason Miller | Score: 28.11 | Experience: 7.0 years
   Skills: Python, PostgreSQL, Redis, Docker
   Reasoning: Matched on Python, PostgreSQL, Redis, Docker; relevant experience found in EXPERIENCE, PROFESSIONAL SUMMARY.

JOB DESCRIPTION: job_description_02.txt
Candidates matched: 3

1. Emily Chen | Score: 100.0 | Experience: 9.0 years
   Skills: React, TypeScript, JavaScript
   Reasoning: Matched on React, TypeScript, JavaScript; relevant experience found in WORK EXPERIENCE, SUMMARY, SKILLS.
2. Aarav Patel | Score: 80.35 | Experience: 4.0 years
   Skills: React, TypeScript, JavaScript
   Reasoning: Matched on React, TypeScript, JavaScript; relevant experience found in SUMMARY, EXPERIENCE, SKILLS.
3. Jason Miller | Score: 73.78 | Experience: 7.0 years
   Skills: React, TypeScript, JavaScript
   Reasoning: Matched on React, TypeScript, JavaScript; relevant experience found in EXPERIENCE, PROFESSIO

In [ ]:
## 3. Retrieval Accuracy

For a small manually labelled evaluation set, check whether the expected good candidate appears in the top-10 results.

In [31]:
from src.hybrid_search import HybridSearcher


evaluation_pairs = [
    ("data/job_descriptions/job_description_01.txt", "Rohan Sharma"),
    ("data/job_descriptions/job_description_02.txt", "Jason Miller"),
    ("data/job_descriptions/job_description_03.txt", "Alexandre Dubois"),
    ("data/job_descriptions/job_description_04.txt", "Vikram Singh Negi"),
    ("data/job_descriptions/job_description_05.txt", "Marcus Vance"),
]

searcher = HybridSearcher()

hits = 0

print("=" * 80)
print("RETRIEVAL ACCURACY")
print("=" * 80)

for jd_path, expected_candidate in evaluation_pairs:
    jd = load_job_description(jd_path)

    results = searcher.search(
        jd,
        top_k=10,
    )

    candidates = []

    for result in results:
        name = result["metadata"]["candidate_name"]

        if name not in candidates:
            candidates.append(name)

    hit = expected_candidate in candidates

    if hit:
        hits += 1

    print()
    print("JD:", jd_path)
    print("Expected:", expected_candidate)
    print("Top-10 contains expected:", hit)

hit_rate = hits / len(evaluation_pairs)

print()
print("=" * 80)
print(f"Hits: {hits}/{len(evaluation_pairs)}")
print(f"Hit rate: {hit_rate:.2%}")
print("=" * 80)

RETRIEVAL ACCURACY

JD: data/job_descriptions/job_description_01.txt
Expected: Rohan Sharma
Top-10 contains expected: True

JD: data/job_descriptions/job_description_02.txt
Expected: Jason Miller
Top-10 contains expected: True

JD: data/job_descriptions/job_description_03.txt
Expected: Alexandre Dubois
Top-10 contains expected: True

JD: data/job_descriptions/job_description_04.txt
Expected: Vikram Singh Negi
Top-10 contains expected: False

JD: data/job_descriptions/job_description_05.txt
Expected: Marcus Vance
Top-10 contains expected: False

Hits: 3/5
Hit rate: 60.00%


In [ ]:
## 4. Latency

Measure embedding generation, vector retrieval, and complete job matching using `time.perf_counter()`.

In [32]:
import time

from src.retrieval.embeddings import EmbeddingService
from src.job_matcher import load_job_description, match_job


print("1. Loading job description...", flush=True)

jd = load_job_description(
    "data/job_descriptions/job_description_01.txt"
)

print("2. Job description loaded.", flush=True)

print("3. Initializing embedding service...", flush=True)

embedding_service = EmbeddingService()

print("4. Embedding service initialized.", flush=True)

print("5. Measuring embedding...", flush=True)

start = time.perf_counter()

embedding_service.embed_text(jd)

embedding_time = time.perf_counter() - start

print(
    f"6. Embedding completed: {embedding_time:.4f} seconds",
    flush=True,
)

print("7. Running complete job matching...", flush=True)

start = time.perf_counter()

result = match_job(jd)

matching_time = time.perf_counter() - start

print(
    f"8. Job matching completed: {matching_time:.4f} seconds",
    flush=True,
)

print()
print("=" * 80)
print("LATENCY RESULTS")
print("=" * 80)

print(f"Embedding latency: {embedding_time:.4f} seconds")
print(f"End-to-end matching latency: {matching_time:.4f} seconds")
print(f"Candidates returned: {len(result['top_matches'])}")

1. Loading job description...
2. Job description loaded.
3. Initializing embedding service...
4. Embedding service initialized.
5. Measuring embedding...
6. Embedding completed: 0.0669 seconds
7. Running complete job matching...
8. Job matching completed: 4.3366 seconds

LATENCY RESULTS
Embedding latency: 0.0669 seconds
End-to-end matching latency: 4.3366 seconds
Candidates returned: 1


In [ ]:
## 5. Analysis

### What worked

The system successfully ingests 30 resumes into a persistent ChromaDB vector store, preserving resume sections and metadata. Hybrid retrieval combines semantic similarity with keyword matching, followed by hard must-have filtering and weighted candidate scoring.

### What did not work perfectly

The matching quality depends on the quality of the extracted skills and the manually curated evaluation labels. Some job descriptions contain skills in natural-language phrases, so deterministic extraction can miss or simplify certain requirements.

### Improvement

A concrete improvement would be to improve job-description skill extraction using a larger curated skill vocabulary and better phrase matching while keeping the metadata extraction deterministic.